# GPU check
Clones the grpo-reasoning repo and checks that the GPU and core libraries work.

In [ ]:
import os, subprocess
REPO = "https://github.com/Vansh8010-boop/grpo-reasoning.git"
os.chdir("/kaggle/working")
if os.path.isdir("grpo-reasoning"):
    subprocess.run(["git", "-C", "grpo-reasoning", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("grpo-reasoning")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi

In [ ]:
import time, torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB  capability {p.major}.{p.minor}")
print("bf16 supported:", torch.cuda.is_bf16_supported())

# fp16 matmul throughput on GPU 0 (T4 has no fast bf16, so fp16 is what we'll train in)
x = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
for _ in range(3): x @ x
torch.cuda.synchronize(); t = time.time()
for _ in range(20): x @ x
torch.cuda.synchronize()
print(f"fp16 matmul: {20 * 2 * 4096**3 / (time.time() - t) / 1e12:.1f} TFLOPS")